In [ ]:
import pandas as pd
import numpy as np

from sklearn.datasets import fetch_openml

In [ ]:
adult = fetch_openml(
    "adult",
    version=2,
    as_frame=True,
    parser="auto"
)

raw_df = adult.frame.copy()

In [ ]:
raw_df.head()

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,class
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,<=50K
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,<=50K
2,28,Local-gov,336951,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,>50K
3,44,Private,160323,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,>50K
4,18,NaN,103497,Some-college,10,Never-married,NaN,Own-child,White,Female,0,0,30,United-States,<=50K


In [ ]:
print("Dataset loaded successfully")
print("Rows:", len(raw_df))
print("Columns:", len(raw_df.columns))
raw_df.info()

Dataset loaded successfully
Rows: 48842
Columns: 15
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48842 entries, 0 to 48841
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype   
---  ------          --------------  -----   
 0   age             48842 non-null  int64   
 1   workclass       46043 non-null  category
 2   fnlwgt          48842 non-null  int64   
 3   education       48842 non-null  category
 4   education-num   48842 non-null  int64   
 5   marital-status  48842 non-null  category
 6   occupation      46033 non-null  category
 7   relationship    48842 non-null  category
 8   race            48842 non-null  category
 9   sex             48842 non-null  category
 10  capital-gain    48842 non-null  int64   
 11  capital-loss    48842 non-null  int64   
 12  hours-per-week  48842 non-null  int64   
 13  native-country  47985 non-null  category
 14  class           48842 non-null  category
dtypes: category(9), int64(6)
memory usage: 2.7 MB


In [ ]:
if raw_df.empty:
    raise ValueError("Pipeline stopped: no data received.")

print("✓ Data availability check passed")

✓ Data availability check passed


In [ ]:
sample_df=pd.DataFrame()
sample_df

""


In [ ]:
if sample_df.empty:
    raise ValueError("Pipeline stopped: no data received.")

print("✓ Data availability check passed")

ValueError: Pipeline stopped: no data received.

In [ ]:
if len(sample_df) == 0:
    raise ValueError("Pipeline stopped: no data received.")

print("✓ Data availability check passed")

ValueError: Pipeline stopped: no data received.

In [ ]:
EXPECTED_COLUMNS = [
    "age",
    "workclass",
    "fnlwgt",
    "education",
    "education-num",
    "marital-status",
    "occupation",
    "relationship",
    "race",
    "sex",
    "capital-gain",
    "capital-loss",
    "hours-per-week",
    "native-country",
    "class"
]

TARGET = "class"

In [ ]:
missing_columns = [
    column
    for column in EXPECTED_COLUMNS
    if column not in raw_df.columns
]

if missing_columns:
    raise ValueError(
        f"Pipeline stopped. Missing columns: {missing_columns}"
    )

print("✓ Expected column check passed")

✓ Expected column check passed


In [ ]:
raw_df1=raw_df.drop("sex",axis=1)

In [ ]:
missing_columns = [
    column
    for column in EXPECTED_COLUMNS
    if column not in raw_df1.columns
]

if missing_columns:
    raise ValueError(
        f"Pipeline stopped. Missing columns: {missing_columns}"
    )

print("✓ Expected column check passed")

ValueError: Pipeline stopped. Missing columns: ['sex']

In [ ]:
print(raw_df[TARGET].value_counts())

class
<=50K    37155
>50K     11687
Name: count, dtype: int64


In [ ]:
if raw_df[TARGET].nunique() < 2 or raw_df[TARGET].nunique() > 2:
    raise ValueError("Pipeline stopped: target has fewer or more than two classes.")

print("✓ Target validation passed")

✓ Target validation passed


In [ ]:
raw_df["sex"].unique()

['Male', 'Female']
Categories (2, object): ['Female', 'Male']

In [ ]:
prepared_df = (
    raw_df
    .drop_duplicates()
    .dropna() #can be replaced with some missing value strategy
    .reset_index(drop=True)
)

print("Prepared dataset shape:", prepared_df.shape)

Prepared dataset shape: (45175, 15)


In [ ]:
assert not prepared_df.empty
assert TARGET in prepared_df.columns
assert prepared_df.isna().sum().sum() == 0
assert prepared_df[TARGET].nunique() == 2

print("✓ Prepared dataset passed output validation")

✓ Prepared dataset passed output validation


Question

You have a raw DataFrame df containing the Adult OpenML dataset. Before starting preprocessing, you want to ensure that the dataset satisfies some basic assumptions.

Write an assert statement to verify that:

The age column contains no negative values.

In [ ]:
assert (raw_df["age"] >= 0).all(), "Age column contains negative values"

In [ ]:
X = raw_df.drop(columns=[TARGET])
y = raw_df["class"]

In [ ]:
numeric_features = X.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_features = X.select_dtypes(
    exclude=["number"]
).columns.tolist()

print("Numeric features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)

Numeric features:
['age', 'fnlwgt', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week']

Categorical features:
['workclass', 'education', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'native-country']


In [ ]:
#Numerical Columns: Missing Value (median), Outlier Treatment (IQR), Scaling (MinMaxscaler)
# These all should be fit+transform on train but directly tranform on test
#Categorical Columns: Missing Value (mode), One hot encoder etc
# These all should be fit+transform on train but directly tranform on test

In [ ]:
#Huge piece of code (multiple tranformation)
#Saving multiple transformation config

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, FunctionTransformer

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
num_cols = X_train.select_dtypes(
    include=np.number
).columns

cat_cols = X_train.select_dtypes(
    exclude=np.number
).columns
print("Numeric features:")
print(num_cols)

print("\nCategorical features:")
print(cat_cols)

Numeric features:
Index(['age', 'fnlwgt', 'education-num', 'capital-gain', 'capital-loss',
       'hours-per-week'],
      dtype='object')

Categorical features:
Index(['workclass', 'education', 'marital-status', 'occupation',
       'relationship', 'race', 'sex', 'native-country'],
      dtype='object')


In [ ]:
def cap_outliers(X):
    X = pd.DataFrame(X).copy()

    Q1 = X.quantile(0.25)
    Q3 = X.quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    return X.clip(lower=lower, upper=upper, axis=1)

In [ ]:
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    #("outlier", FunctionTransformer(cap_outliers)),
    ("scaler", MinMaxScaler()) #scaling 0-1
])

In [ ]:
cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),#mode
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

In [ ]:
preprocessor = ColumnTransformer([
    ("num", num_pipeline, num_cols),
    ("cat", cat_pipeline, cat_cols)
])

In [ ]:
preprocessor

ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', MinMaxScaler())]),
                                 Index(['age', 'fnlwgt', 'education-num', 'capital-gain', 'capital-loss',
       'hours-per-week'],
      dtype='object')),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('encoder',
                                                  OneHotEncoder(handle_unknown='ignore'))]),
                                 Index(['workclass', 'education', 'marital-status', 'occupation',
       'relationship', 'race', 'sex', 'native-country'],
      dtype='object'))])

In [ ]:
X_train_transformed = preprocessor.fit_transform(X_train)

In [ ]:
X_test_transformed = preprocessor.transform(X_test)

In [ ]:
X_train_transformed.shape

(39073, 105)

In [ ]:
import joblib #or pickle

joblib.dump(preprocessor, "preprocessor.pkl")

['preprocessor.pkl']

In [ ]:
#this extra step only for colab
from google.colab import files

files.download("preprocessor.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
#this extra step only for colab
from google.colab import files

uploaded = files.upload()

Saving preprocessor.pkl to preprocessor (1).pkl


In [ ]:
preprocessor1 = joblib.load("preprocessor.pkl")

In [ ]:
preprocessor.get_feature_names_out()

array(['num__age', 'num__fnlwgt', 'num__education-num',
       'num__capital-gain', 'num__capital-loss', 'num__hours-per-week',
       'cat__workclass_Federal-gov', 'cat__workclass_Local-gov',
       'cat__workclass_Never-worked', 'cat__workclass_Private',
       'cat__workclass_Self-emp-inc', 'cat__workclass_Self-emp-not-inc',
       'cat__workclass_State-gov', 'cat__workclass_Without-pay',
       'cat__education_10th', 'cat__education_11th',
       'cat__education_12th', 'cat__education_1st-4th',
       'cat__education_5th-6th', 'cat__education_7th-8th',
       'cat__education_9th', 'cat__education_Assoc-acdm',
       'cat__education_Assoc-voc', 'cat__education_Bachelors',
       'cat__education_Doctorate', 'cat__education_HS-grad',
       'cat__education_Masters', 'cat__education_Preschool',
       'cat__education_Prof-school', 'cat__education_Some-college',
       'cat__marital-status_Divorced',
       'cat__marital-status_Married-AF-spouse',
       'cat__marital-status_Married-civ

In [ ]:
preprocessor.feature_names_in_

array(['age', 'workclass', 'fnlwgt', 'education', 'education-num',
       'marital-status', 'occupation', 'relationship', 'race', 'sex',
       'capital-gain', 'capital-loss', 'hours-per-week', 'native-country'],
      dtype=object)

In [ ]:
from feature_engine.outliers import Winsorizer
num_iqr_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("outlier", Winsorizer(
        capping_method="iqr",
        tail="both",
        fold=1.5
    )),
    ("scaler", MinMaxScaler())
])

num_normal_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", MinMaxScaler())
])